###  Main problem statement: “To determine which client segments generate the highest gross profit while maintaining strong customer satisfaction” (enabling the company to prioritize high-value clients, improve client retention, and support sustainable business growth)​

#### Key points (subproblems)​

#### Profitability and its drivers by client segment: Analyse gross profit and gross margin across client type, industry sector, organisation size and location, while examining hardware, software and manpower costs and service ratings to identify high-value segments and opportunities for cost optimisation and margin improvement.​

In [1]:
import pandas as pd
import plotly.express as px

# ============================================
# Load and Prepare Data
# ============================================
xls = pd.ExcelFile("merged_cleaned.xlsx")
df_merged = pd.read_excel(xls, xls.sheet_names[0])

# Financial calculations
df_merged["COGS"] = (
    df_merged["HARDWARE"]
    + df_merged["SOFTWARE"]
    + df_merged["MANPOWER"]
)

df_merged["GROSS_PROFIT"] = (
    df_merged["REVENUE"]
    - df_merged["COGS"]
)

df_merged["GROSS_MARGIN"] = (
    df_merged["GROSS_PROFIT"]
    / df_merged["REVENUE"]
) * 100

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

frames = []

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)
        .agg({
            "GROSS_PROFIT": "sum",
            "GROSS_MARGIN": "mean",
            "REVENUE": "sum",
            "NPS RATING": "mean"
        })
        .reset_index()
    )

    grouped = grouped.rename(columns={col: "Segment"})
    grouped["Segmentation"] = label

    # Sort from highest to lowest profit
    grouped = grouped.sort_values(
        "GROSS_PROFIT",
        ascending=False
    )

    frames.append(grouped)

plot_df = pd.concat(frames, ignore_index=True)

# ============================================
# Animated Horizontal Bar Chart
# ============================================

fig = px.bar(

    plot_df,

    x="GROSS_PROFIT",

    y="Segment",

    orientation="h",

    color="GROSS_MARGIN",

    color_continuous_scale="Viridis",

    animation_frame="Segmentation",

    hover_name="Segment",

    hover_data={
        "GROSS_PROFIT": ":,.0f",
        "GROSS_MARGIN": ":.2f",
        "REVENUE": ":,.0f",
        "NPS RATING": ":.2f"
    },

    title="Gross Profit Across Client Segmentations"

)

fig.update_layout(

    template="plotly_white",

    title_x=0.5,

    xaxis_title="Total Gross Profit",

    yaxis_title="Client Segment",

    height=650,

    coloraxis_colorbar=dict(
        title="Gross Margin (%)"
    )

)

fig.show()
fig1 = fig

## 1. Client-type profitability

### Corrected-data findings

- **Private clients** contribute **$50.29 million** of gross profit from **106 clients**, with an average gross margin of **45.0%**.
- **Government clients** contribute **$9.14 million** from **21 clients**, with an average gross margin of **44.4%**.
- **NPO clients** contribute **$0.26 million** from **13 clients**, with an average gross margin of **18.5%**. The low contribution and margin should be interpreted alongside the small client base.

### Implication

Private clients are the main commercial value pool in this cleaned client-year dataset. Government is a smaller but relatively efficient segment, while NPO accounts warrant a separate cost-to-serve and pricing review rather than comparison only on total profit.

## 2. Sector profitability

### Corrected-data findings

- **Healthcare** is the largest sector by gross profit at **$14.72 million** across **32 clients**.
- **Info Tech** contributes **$12.56 million** and **Manufacturing** contributes **$10.74 million**; both have average gross margins of approximately **45.3%**.
- **Transportation** has the highest average sector margin at **46.1%**, but contributes **$4.63 million** across **10 clients**, so its efficiency is not the same as portfolio scale.

### Implication

Healthcare, Info Tech and Manufacturing are the main sector-level value pools. Transportation is a smaller high-margin segment that may merit investigation for scalable practices, without assuming that the observed margin difference is causal.

## 3. Organisation size and country

- Clients with **50–200 staff** contribute **$29.56 million** of gross profit, while clients with **more than 200 staff** contribute **$24.38 million**.
- The **1–49 staff** group contributes **$5.75 million** and has the lowest average margin at **32.7%**, despite an average service score of approximately **4.06/5**.
- **Singapore** contributes **$44.14 million**, while **Malaysia** contributes **$9.60 million** with a higher average margin (**46.5% versus 41.2%**).
- **China** has the highest average country margin (**50.5%**) but only four clients, so the result is less stable than the larger-country comparisons.

These findings describe differences in the observed records. They do not prove that organisation size, country or service quality causes profitability.


In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ============================================
# Create Average Service Rating
# ============================================
service_cols = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT",
]

df_merged["AVG_SERVICE"] = df_merged[service_cols].mean(axis=1)

# ============================================
# Segmentations
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY",
}

aggregated = {}

for label, col in segmentations.items():
    grouped = (
        df_merged.groupby(col)
        .agg(
            {
                "AVG_SERVICE": "mean",
                "GROSS_MARGIN": "mean",
                "GROSS_PROFIT": "sum",
                "REVENUE": "sum",
            }
        )
        .reset_index()
    )

    grouped = grouped.rename(columns={col: "Segment"})
    grouped["Segmentation"] = label
    aggregated[label] = grouped

# Combined view for the All option
all_segments = []
for label, df in aggregated.items():
    temp = df[
        ["Segment", "AVG_SERVICE", "GROSS_MARGIN", "GROSS_PROFIT", "REVENUE"]
    ].copy()
    temp["Segmentation"] = label
    all_segments.append(temp)

aggregated["All"] = pd.concat(all_segments, ignore_index=True)

# Global Min/Max for unified color scale across view switches
cmin_gp = df_merged["GROSS_PROFIT"].min()
cmax_gp = df_merged["GROSS_PROFIT"].sum()

# ============================================
# Default View
# ============================================
data = aggregated["All"]
segment_col = "Segment"

bubble_size = (data["REVENUE"] / data["REVENUE"].max()) * 60 + 12

# ============================================
# Initial Figure
# ============================================
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=data["AVG_SERVICE"],
        y=data["GROSS_MARGIN"],
        mode="markers+text",
        text=data[segment_col],
        textposition="top center",
        marker=dict(
            size=bubble_size,
            color=data["GROSS_PROFIT"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Gross Profit"),
            sizemode="diameter",
            line=dict(width=1),
        ),
        customdata=np.stack(
            (data[segment_col], data["GROSS_PROFIT"], data["REVENUE"]), axis=-1
        ),
        hovertemplate="<b>%{customdata[0]}</b><br><br>"
        + "Average Service Rating: %{x:.2f}<br>"
        + "Gross Margin: %{y:.2f}%<br>"
        + "Gross Profit: %{customdata[1]:,.0f}<br>"
        + "Revenue: %{customdata[2]:,.0f}<extra></extra>",
    )
)

# Dropdown
buttons = []

# Build dropdown items for ["All"] + individual segmentations
dropdown_keys = ["All"] + list(segmentations.keys())

for label in dropdown_keys:
    temp = aggregated[label]

    b_size = (temp["REVENUE"] / temp["REVENUE"].max()) * 60 + 12

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {
                    "x": [temp["AVG_SERVICE"]],
                    "y": [temp["GROSS_MARGIN"]],
                    "text": [temp["Segment"]],
                    "customdata": [
                        np.stack(
                            (
                                temp["Segment"],
                                temp["GROSS_PROFIT"],
                                temp["REVENUE"],
                            ),
                            axis=-1,
                        )
                    ],
                    "marker": [
                        dict(
                            size=b_size,
                            color=temp["GROSS_PROFIT"],
                            colorscale="Viridis",
                            showscale=True,
                            colorbar=dict(title="Gross Profit"),
                            sizemode="diameter",
                            line=dict(width=1),
                        )
                    ],
                },
                {
                    "title": f"Service Quality vs Gross Margin ({label})",
                    "xaxis": {"title": "Average Service Rating"},
                    "yaxis": {"title": "Average Gross Margin (%)"},
                },
            ],
        )
    )

# ============================================
# Layout Configuration
# ============================================
fig.update_layout(
    title="Service Quality vs Gross Margin (All Segments)",
    template="plotly_white",
    xaxis_title="Average Service Rating",
    yaxis_title="Average Gross Margin (%)",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            y=1.15,
            xanchor="left",
            yanchor="top",
            showactive=True,
        )
    ],
)

fig.show()

## Service quality versus profitability

The corrected dataset records the highest average overall service score for **Finance (4.25/5)**, but Finance contains only one client. Among larger sectors, **Transportation (4.11/5)** and **Security (4.05/5)** combine relatively high service scores with gross margins of **46.1%** and **43.3%** respectively.

**Manufacturing** contributes substantial gross profit (**$10.74 million**) but has a lower average service score (**3.86/5**), making it a useful segment for investigating whether service delivery practices differ from stronger-rated segments. This is an association-based observation, not evidence that service scores determine margin.

The matrix should be read with revenue scale, gross profit and client count together. A small segment can appear strong on an average while contributing little total value.


In [3]:
import pandas as pd
import plotly.graph_objects as go

# ============================================
# Segmentations & Data Aggregation
# ============================================
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY",
}

aggregated = {}

# Individual Segmentations
for label, col in segmentations.items():
    grouped = (
        df_merged.groupby(col)[["HARDWARE", "SOFTWARE", "MANPOWER"]]
        .sum()
        .reset_index()
    )
    grouped = grouped.rename(columns={col: "Segment"})
    aggregated[label] = grouped

# Combined "All" View across all segment categories
all_segments = []
for label, df in aggregated.items():
    temp = df[["Segment", "HARDWARE", "SOFTWARE", "MANPOWER"]].copy()
    all_segments.append(temp)

aggregated["All"] = pd.concat(all_segments, ignore_index=True)

# ============================================
# Default Chart View ("All")
# ============================================
default = "All"
data = aggregated[default]

fig = go.Figure()

fig.add_trace(
    go.Bar(x=data["Segment"], y=data["HARDWARE"], name="Hardware")
)

fig.add_trace(
    go.Bar(x=data["Segment"], y=data["SOFTWARE"], name="Software")
)

fig.add_trace(
    go.Bar(x=data["Segment"], y=data["MANPOWER"], name="Manpower")
)

# ============================================
# Dropdown Options Setup
# ============================================
buttons = []
dropdown_keys = ["All"] + list(segmentations.keys())

for label in dropdown_keys:
    temp = aggregated[label]

    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {
                    "x": [
                        temp["Segment"],
                        temp["Segment"],
                        temp["Segment"],
                    ],
                    "y": [
                        temp["HARDWARE"],
                        temp["SOFTWARE"],
                        temp["MANPOWER"],
                    ],
                },
                {
                    "title": f"Cost Composition by {label}",
                    "xaxis": {"title": label if label != "All" else "Segments"},
                },
            ],
        )
    )

mode_buttons = [
    dict(
        label="Stacked",
        method="relayout",
        args=[{"barmode": "stack"}],
    ),
    dict(
        label="Grouped",
        method="relayout",
        args=[{"barmode": "group"}],
    ),
]

# ============================================
# Layout
# ============================================
fig.update_layout(
    title="Cost Composition by All Segments",
    xaxis_title="Segments",
    yaxis_title="Total Cost",
    barmode="stack",
    template="plotly_white",
    height=550,
    autosize=True,
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            y=1.15,
            xanchor="left",
            yanchor="top",
            showactive=True,
        ),
        dict(
            buttons=mode_buttons,
            direction="right",
            x=0.55,
            y=1.15,
            xanchor="left",
            yanchor="top",
            showactive=True,
        ),
    ],
)

fig.show()
fig3 = fig

## Cost composition: Hardware, Software and Manpower

Across the cleaned dataset, total costs are:

- **Manpower:** $32.60 million
- **Software:** $26.95 million
- **Hardware:** $15.83 million

Manpower is therefore the largest cost component overall. The pattern is also visible in the larger value pools: Private clients have **$26.67 million** of manpower cost compared with **$23.08 million** of software cost and **$13.10 million** of hardware cost.

For organisation size, the **more-than-200 staff** group has the largest manpower cost (**$15.42 million**), while the **50–200 staff** group has the largest total gross-profit contribution. Cost totals should be compared with revenue scale and gross margin; a large cost amount is not automatically inefficient.


In [4]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Dataset Initialization
# ---------------------------------------------------------
df = df_merged.copy()
if 'AVG_SERVICE' not in df.columns:
    df['AVG_SERVICE'] = df[['PRESALES AND PARTNERSHIP', 'TECHNICAL EXPERTISE', 'PROJECT DELIVERY', 'POST-SALES SUPPORT']].mean(axis=1)

# Dimension Mapping
segmentation_opts = {
    'Industry Sector': 'SECTOR',
    'Client Type': 'TYPE',
    'Organisation Size': 'STAFF STRENGTH',
    'Country': 'COUNTRY'
}

# ---------------------------------------------------------
# 2. Dash Layout
# ---------------------------------------------------------
app = Dash(__name__)
app.title = 'Client Profitability & Cost Intelligence'

logo_mark = html.Div('$', style={
    'width': '40px', 'height': '40px', 'borderRadius': '8px',
    'backgroundColor': '#2B6CB0', 'color': 'white', 'fontSize': '22px',
    'fontWeight': '700', 'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'
})

app.layout = html.Div([
    # Top Header
    html.Div([
        html.Div([
            logo_mark,
            html.Div([
                html.Div('STRATEGIC FINANCIAL ANALYTICS', style={'fontSize': '11px', 'fontWeight': '800', 'letterSpacing': '1.2px', 'color': '#2B6CB0'}),
                html.Div('Client Segment Profitability & Performance Optimization', style={'fontSize': '18px', 'fontWeight': '700', 'color': '#1A202C'})
            ])
        ], style={'display': 'flex', 'alignItems': 'center', 'gap': '12px'})
    ], style={'padding': '14px 28px', 'backgroundColor': 'white', 'borderBottom': '1px solid #E2E8F0'}),

    # Control Toolbar
    html.Div([
        html.Div([
            html.Label('Primary Segment Dimension', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Dropdown(id='dim-control', options=[{'label': k, 'value': v} for k, v in segmentation_opts.items()], value='SECTOR', clearable=False)
        ]),
        html.Div([
            html.Label('Year', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Dropdown(id='year-control', options=[{'label': 'All Years', 'value': 'All'}] + [{'label': str(y), 'value': y} for y in sorted(df['YEAR'].unique())], value='All', clearable=False)
        ]),
        html.Div([
            html.Label('Country Filter', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Dropdown(id='country-control', options=[{'label': 'All Countries', 'value': 'All'}] + [{'label': c, 'value': c} for c in sorted(df['COUNTRY'].unique())], value='All', clearable=False)
        ]),
        html.Div([
            html.Label('Min NPS Score Filter', style={'fontWeight': '700', 'fontSize': '12px', 'color': '#2D3748'}),
            dcc.Slider(id='nps-control', min=1, max=10, step=1, value=1, marks={i: str(i) for i in range(1, 11)})
        ])
    ], style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr 1fr 1.5fr', 'gap': '16px', 'padding': '14px 28px', 'backgroundColor': '#F7FAFC', 'borderBottom': '1px solid #E2E8F0'}),

    # Dynamic KPI Summary Section
    html.Div(id='kpi-container', style={'display': 'grid', 'gridTemplateColumns': 'repeat(4, 1fr)', 'gap': '16px', 'padding': '16px 28px 0 28px', 'backgroundColor': '#EDF2F7'}),

    # Main Grid Layout for All 3 Charts
    html.Div([
        # Row 1: Profitability & Cost Structures (sbs)
        html.Div([
            html.Div([dcc.Graph(id='chart-1-profitability')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'}),
            html.Div([dcc.Graph(id='chart-2-costs')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'}),
        ], style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '18px'}),

        # Row 2: Service Quality & NPS Matrix
        html.Div([
            html.Div([dcc.Graph(id='chart-3-satisfaction')], style={'backgroundColor': 'white', 'borderRadius': '8px', 'padding': '10px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'})
        ])
    ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '18px', 'padding': '16px 28px 24px 28px', 'backgroundColor': '#EDF2F7'}),

    # Global Footer
    html.Footer('Enterprise Operations Analytics • Internal Decision Support Dashboard', style={'padding': '12px 28px', 'fontSize': '11px', 'color': '#718096', 'backgroundColor': 'white', 'borderTop': '1px solid #E2E8F0'})
], style={'fontFamily': 'Segoe UI, Arial, sans-serif', 'backgroundColor': '#EDF2F7', 'minHeight': '100vh'})


# ---------------------------------------------------------
# 3. Unified Callback
# ---------------------------------------------------------
@app.callback(
    [
        Output('kpi-container', 'children'),
        Output('chart-1-profitability', 'figure'),
        Output('chart-2-costs', 'figure'),
        Output('chart-3-satisfaction', 'figure')
    ],
    [
        Input('dim-control', 'value'),
        Input('year-control', 'value'),
        Input('country-control', 'value'),
        Input('nps-control', 'value')
    ]
)
def update_dashboard_view(dimension, year_val, country_val, min_nps):
    # Filters
    d = df.copy()
    if year_val != 'All':
        d = d[d['YEAR'] == year_val]
    if country_val != 'All':
        d = d[d['COUNTRY'] == country_val]
    d = d[d['NPS RATING'] >= min_nps]

    # KPIs
    if not d.empty:
        total_rev = d['REVENUE'].sum()
        total_profit = d['GROSS_PROFIT'].sum()
        avg_margin = d['GROSS_MARGIN'].mean() * 100
        avg_service = d['AVG_SERVICE'].mean()
    else:
        total_rev, total_profit, avg_margin, avg_service = 0, 0, 0, 0

    kpi_cards = [
        html.Div([
            html.Div(title, style={'fontSize': '11px', 'color': '#718096', 'fontWeight': '600'}),
            html.Div(val, style={'fontSize': '20px', 'fontWeight': '700', 'color': '#2B6CB0', 'marginTop': '4px'})
        ], style={'backgroundColor': 'white', 'padding': '14px 18px', 'borderRadius': '8px', 'boxShadow': '0 1px 3px rgba(0,0,0,0.05)'})
        for title, val in [
            ("Total Revenue", f"${total_rev:,.0f}"),
            ("Total Gross Profit", f"${total_profit:,.0f}"),
            ("Avg Gross Margin", f"{avg_margin:.1f}%"),
            ("Avg Service Score", f"{avg_service:.1f} / 10")
        ]
    ]

    if d.empty:
        empty_fig = go.Figure().update_layout(title="No clients match the selected filters.", template="plotly_white")
        return kpi_cards, empty_fig, empty_fig, empty_fig

    # ---------------------------------------------------------
    # CHART 1: Gross Profit & Margin Efficiency
    # ---------------------------------------------------------
    agg_profit = d.groupby(dimension)[['GROSS_PROFIT', 'REVENUE', 'GROSS_MARGIN']].agg({
        'GROSS_PROFIT': 'sum',
        'REVENUE': 'sum',
        'GROSS_MARGIN': 'mean'
    }).reset_index().sort_values('GROSS_PROFIT', ascending=True)

    fig1 = px.bar(
        agg_profit,
        x='GROSS_PROFIT',
        y=dimension,
        color='GROSS_MARGIN',
        color_continuous_scale='Viridis',
        orientation='h',
        template='plotly_white',
        title=f'1. Total Gross Profit & Gross Margin % by {dimension}'
    )
    fig1.update_layout(
        height=420,
        margin=dict(t=50, l=110, r=30, b=40),
        coloraxis_colorbar=dict(title="Margin %", tickformat=".0%")
    )
    fig1.update_xaxes(tickformat='$,.0f', title='Gross Profit ($)')
    fig1.update_yaxes(title='')

    # ---------------------------------------------------------
    # CHART 2: Cost Structure Breakdown
    # ---------------------------------------------------------
    agg_cost = d.groupby(dimension)[['HARDWARE', 'SOFTWARE', 'MANPOWER']].sum().reset_index()
    
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=agg_cost[dimension], y=agg_cost['HARDWARE'], name='Hardware', marker_color='#3182CE'))
    fig2.add_trace(go.Bar(x=agg_cost[dimension], y=agg_cost['SOFTWARE'], name='Software', marker_color='#DD6B20'))
    fig2.add_trace(go.Bar(x=agg_cost[dimension], y=agg_cost['MANPOWER'], name='Manpower', marker_color='#38A169'))
    
    fig2.update_layout(
        title=f'2. Cost Composition Breakdown by {dimension}',
        barmode='stack',
        template='plotly_white',
        height=420,
        margin=dict(t=50, l=60, r=30, b=40),
        xaxis_title='',
        yaxis_title="Cost ($)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    fig2.update_yaxes(tickformat='$,.0f')

    # ---------------------------------------------------------
    # CHART 3: Service Quality vs. Profitability Matrix
    # ---------------------------------------------------------
    agg_satisfaction = d.groupby(dimension).agg({
        'AVG_SERVICE': 'mean',
        'GROSS_MARGIN': 'mean',
        'REVENUE': 'sum',
        'NPS RATING': 'mean'
    }).reset_index()

    fig3 = go.Figure()
    
    for seg in agg_satisfaction[dimension]:
        sub = agg_satisfaction[agg_satisfaction[dimension] == seg]
        fig3.add_trace(go.Scatter(
            x=sub['AVG_SERVICE'],
            y=sub['GROSS_MARGIN'] * 100,
            mode='markers+text',
            name=str(seg),
            text=sub[dimension],
            textposition="top center",
            marker=dict(
                size=np.clip(np.sqrt(sub['REVENUE'] / 50000) * 8, 14, 45),
                opacity=0.8,
                line=dict(width=1, color='white')
            ),
            customdata=np.stack((sub['REVENUE'], sub['NPS RATING']), axis=-1),
            hovertemplate="<b>%{text}</b><br>" +
                          "Avg Service Score: %{x:.2f} / 10<br>" +
                          "Gross Margin: %{y:.1f}%<br>" +
                          "Total Revenue: $%{customdata[0]:,.0f}<br>" +
                          "Avg NPS Rating: %{customdata[1]:.1f}<extra></extra>"
        ))

    # Median Target Lines
    fig3.add_vline(x=agg_satisfaction['AVG_SERVICE'].median(), line_dash='dash', line_color='#A0AEC0',
                   annotation_text='Median Quality', annotation_position='top right')
    fig3.add_hline(y=agg_satisfaction['GROSS_MARGIN'].median() * 100, line_dash='dash', line_color='#A0AEC0',
                   annotation_text='Median Margin', annotation_position='bottom right')

    fig3.update_layout(
        title=f'3. Service Quality Score vs Gross Margin (%) by {dimension} (Bubble Size = Total Revenue)',
        template='plotly_white',
        height=450,
        margin=dict(t=50, l=60, r=30, b=40),
        xaxis=dict(title='Average Service Score (1–10)'),
        yaxis=dict(title='Gross Margin (%)'),
        showlegend=True
    )

    return kpi_cards, fig1, fig2, fig3


# Presentation launch command — run manually outside notebook execution.
# app.run(debug=True)

# Strategic analysis from the corrected cleaned dataset

## 1. Where is the commercial value concentrated?

Healthcare, Info Tech and Manufacturing are the three largest sectors by gross profit, contributing **$14.72 million**, **$12.56 million** and **$10.74 million** respectively. At client-type level, Private clients contribute **$50.29 million**, substantially more than Government (**$9.14 million**) and NPO (**$0.26 million**).

## 2. Where should margin improvement be investigated?

The 1–49 staff group has the lowest observed average gross margin (**32.7%**) despite a relatively high service score (**4.06/5**). NPO has an even lower average margin (**18.5%**), but its contribution is based on only 13 clients. These groups are candidates for reviewing pricing, delivery effort and cost-to-serve.

Singapore is the largest country-level value pool (**$44.14 million**) but has a lower average margin (**41.2%**) than Malaysia (**46.5%**) and China (**50.5%**). China has only four clients, so its margin result should not be generalised.

## 3. What cost driver deserves attention?

Manpower is the largest aggregate cost component at **$32.60 million**, followed by software at **$26.95 million** and hardware at **$15.83 million**. Operational reviews should therefore examine resource allocation and delivery effort, particularly in the Private and more-than-200-staff groups. The charts do not establish that reducing manpower would necessarily increase satisfaction or gross profit.

## 4. Does service quality cause profitability?

No causal conclusion can be drawn from these descriptive charts. The output shows segment differences: Manufacturing has substantial gross profit but a lower average service score than Transportation, while Finance has the highest average service score but only one client. These patterns support further investigation, not a claim that satisfaction causes margin performance.
